In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix,classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
sns.set_style("darkgrid")


In [ ]:
#Reading the data file
df = pd.read_csv('kidney_disease.csv')

df.head(5)

In [ ]:
df = df.drop('id', axis=1)

df.shape, df.describe

In [ ]:
df.info()

In [ ]:
missing_values = df.isnull().sum()

missing_values

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
# Assuming your DataFrame is named 'df'
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0] # Filter features with?

# Create a bar chart
plt.figure(figsize=(10, 6))
missing_values.plot(kind='bar')
plt.title('Missing Values per Feature')
plt.xlabel('Features')
plt.ylabel('Count of Missing Values')
plt.xticks(rotation=45) # Rotate x-axis labels for readability
# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
# Filling missing values for numerical features
# Numerical columns for median imputation
numerical_columns = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod','pot', 'hemo']
# Impute missing values with the median
df[numerical_columns] = df[numerical_columns].fillna(df[numerical_columns].median())
# Check if there are any remaining missing values
remaining_missing_values = df.isnull().sum()
# Display the columns with missing values and their counts (after imputation)
remaining_missing_values[remaining_missing_values > 0]


In [ ]:
categorical_columns = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet','pe', 'ane','pcv','wc','rc']
# Impute missing values with the mode
df[categorical_columns] = df[categorical_columns].apply(lambda x: x.fillna(x.mode()[0]))
# Check if there are any remaining missing values
remaining_missing_values_categorical = df[categorical_columns].isnull().sum()
# Display the columns with missing values and their counts (after imputation)
remaining_missing_values_categorical[remaining_missing_values_categorical > 0]

In [ ]:
 # Check for missing values in each column
missing_values = df.isnull().sum()
# Display the count of missing values for each column
print(missing_values)


In [ ]:
# Specify numerical columns for outlier detection
numerical_columns = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod','pot', 'hemo']
# Create subplots for outlier visualization
plt.figure(figsize=(16, 10))
for i, column in enumerate(numerical_columns, 1):
    plt.subplot(4, 3, i)
    sns.boxplot(data=df, x=column)
    plt.title(f'Box Plot of {column}')
    plt.tight_layout()
    plt.show()
# Define a function to identify outliers using IQR method
def identify_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers
# Detect and display outliers for each numerical column
for column in numerical_columns:
    outliers = identify_outliers_iqr(df, column)
    if not outliers.empty:
        print(f'Outliers in {column}:')
        print(outliers)


In [ ]:
# Define a function to handle outliers using IQR method
def handle_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[column] = np.where((data[column] < lower_bound) | (data[column] > upper_bound), data[column].median(), data[column])
# Specify numerical columns for outlier handling
numerical_columns = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod','pot', 'hemo']
# Handle outliers for each numerical column
for column in numerical_columns:
    handle_outliers_iqr(df, column)

In [ ]:
categorical_columns = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
# One-hot encode categorical columns
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

In [ ]:
df_encoded.head()

In [ ]:

plt.figure(figsize=(8, 6))
sns.countplot(x='classification', data=df_encoded)
plt.title('Distribution of the Target Variable')
plt.xlabel('Classification (0: notckd, 1: ckd)')
plt.ylabel('Count')
plt.show()



In [ ]:
!pip install  imbalanced-learn


In [ ]:
# Define a mapping dictionary
mapping = {'notckd': 0, 'ckd': 1}
# Map the values in the 'classification' column of df_encoded using the defined mapping
df_encoded['classification'] = df_encoded['classification'].map(mapping)
# Display the first few rows of the updated dataframe
df_encoded.head()


In [ ]:
import numpy as np

# Replace non-numeric values with NaN
df_encoded.replace(r'\t?', 1, regex=True, inplace=True)

# Optionally, fill NaN values or drop rows/columns with NaN
# Example: Fill NaN values with column mean
df_encoded.fillna(df_encoded.mean(), inplace=True)

# Ensure all data is numeric
df_encoded = df_encoded.apply(pd.to_numeric, errors='coerce')


In [ ]:
df_encoded['classification'].value_counts()

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
# Separate features (X) and target variable (y)
X = df_encoded.drop('classification', axis=1)

y = df_encoded['classification']

y.fillna(y.mode()[0], inplace=True)

# Ensure the target variable is updated in the original DataFrame
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Apply SMOTE to the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
# Check the class distribution after resampling
print("Class distribution after SMOTE:")
print(y_train_resampled.value_counts())




In [ ]:
from sklearn.preprocessing import MinMaxScaler
# Instantiate the MinMaxScaler
scaler = MinMaxScaler()
# Fit and transform the scaler on the training data
X_train_normalized = scaler.fit_transform(X_train_resampled)
# Transform the test data using the same scaler
X_test_normalized = scaler.transform(X_test)
# Convert the normalized arrays back to DataFrames (optional)
X_train_normalized = pd.DataFrame(X_train_normalized, columns=X.columns)
X_test_normalized = pd.DataFrame(X_test_normalized, columns=X.columns)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# Instantiate the Random Forest model
rf_model = RandomForestClassifier(random_state=42)
# Fit the model to the normalized training data
rf_model.fit(X_train_normalized, y_train_resampled)
# Predictions on the test set
y_pred_rf = rf_model.predict(X_test_normalized)
# Evaluate the performance
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy:", accuracy_rf)
# Print classification report and confusion matrix
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
from sklearn.svm import SVC
# Instantiate the Support Vector Machine model
svm_model = SVC(random_state=42)
# Fit the model to the normalized training data
svm_model.fit(X_train_normalized, y_train_resampled)
# Predictions on the test set
y_pred_svm = svm_model.predict(X_test_normalized)
# Evaluate the performance
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("SVM Accuracy:", accuracy_svm)
# Print classification report and confusion matrix
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# Instantiate the Random Forest model
rf_model = RandomForestClassifier(random_state=42)
# Fit the model to the normalized training data
rf_model.fit(X_train_normalized, y_train_resampled)
# Predictions on the test set
y_pred_rf = rf_model.predict(X_test_normalized)
# Evaluate the performance
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy:", accuracy_rf)
# Print classification report and confusion matrix
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
from sklearn.neural_network import MLPClassifier
# Instantiate the Artificial Neural Network model
ann_model = MLPClassifier(random_state=42)
# Fit the model to the normalized training data
ann_model.fit(X_train_normalized, y_train_resampled)
# Predictions on the test set
y_pred_ann = ann_model.predict(X_test_normalized)
# Evaluate the performance
accuracy_ann = accuracy_score(y_test, y_pred_ann)
print("ANN Accuracy:", accuracy_ann)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ann))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ann))


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
# Instantiate the ANN model with a different architecture
ann_model_custom = MLPClassifier(hidden_layer_sizes=(128, 64),activation='relu', solver='adam', random_state=42)
# Fit the model on the normalized and resampled training data
ann_model_custom.fit(X_train_normalized, y_train_resampled)
# Predict on the normalized test data
y_pred_ann_custom = ann_model_custom.predict(X_test_normalized)
# Evaluate the performance of the ANN with the new architecture
accuracy_ann_custom = accuracy_score(y_test, y_pred_ann_custom)
print("Custom ANN Accuracy:", accuracy_ann_custom)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_ann_custom))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ann_custom))
# Plot the confusion matrix for the custom architecture
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_ann_custom), annot=True, fmt="d",cmap="Blues", cbar=False)
plt.title("Confusion Matrix - Custom ANN Architecture")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
from sklearn.tree import DecisionTreeClassifier
# Instantiate the Decision Tree model
decision_tree_model = DecisionTreeClassifier(random_state=42)
# Fit the model on the normalized and resampled training data
decision_tree_model.fit(X_train_normalized, y_train_resampled)
# Predict on the normalized test data
y_pred_decision_tree = decision_tree_model.predict(X_test_normalized)
# Evaluate the performance of the Decision Tree
accuracy_decision_tree = accuracy_score(y_test, y_pred_decision_tree)
print("Decision Tree Accuracy:", accuracy_decision_tree)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_decision_tree))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_decision_tree))
# Plot the confusion matrix for the Decision Tree
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_decision_tree), annot=True,fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix - Decision Tree")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# Instantiate the K-Nearest Neighbors model
knn_model = KNeighborsClassifier()
# Fit the model on the normalized and resampled training data
knn_model.fit(X_train_normalized, y_train_resampled)
# Predict on the normalized test data
y_pred_knn = knn_model.predict(X_test_normalized)
# Evaluate the performance of K-Nearest Neighbors
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print("K-NN Accuracy:", accuracy_knn)
print("\nClassification Report:")

print(classification_report(y_test, y_pred_knn))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))
# Plot the confusion matrix for K-Nearest Neighbors
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_knn), annot=True, fmt="d",cmap="Blues", cbar=False)
plt.title("Confusion Matrix - K-NN")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
models = ['SVM', 'Decision Tree', 'K-NN', 'ANN']
# List of accuracy scores for each model
accuracies = [accuracy_svm, accuracy_decision_tree, accuracy_knn, accuracy_ann]
# Create a bar plot
plt.figure(figsize=(10, 6))
plt.bar(models, accuracies, color=['blue', 'green', 'orange', 'red'])
plt.title('Model Comparison - Accuracy')
plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.ylim(0, 1) # Set y-axis limit to the range of accuracy scores (0 to 1)
plt.show()


In [ ]:
import pickle
file_name = 'model.pkl'

pickle.dump(rf_model, open(file_name, 'wb'))